# Train a FlyDataClassifier on `make_moons`

This minimal example maps two numerical features into an L2/Tm1 circuit selected from MaleCNS v1.0, then trains the circuit and classification head with full-batch gradient descent.

In [ ]:
from pathlib import Path

import jax.numpy as jnp
import numpy as np
import optax
import trackio
from flax import nnx
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import trange

from flyx.core import CircuitSpec, NeuronQuery as Q
from flyx.model import FlyConfig, FlyDataClassifier, FlyModel

In [ ]:
x, y = make_moons(n_samples=2_000, noise=0.15, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype(np.float32)
x_test = scaler.transform(x_test).astype(np.float32)
y_train = y_train.astype(np.int32)
y_test = y_test.astype(np.int32)

In [ ]:
circuit = CircuitSpec(
    neurons=(
        Q.field("superclass").eq("ol_intrinsic")
        & Q.field("type").isin(["L2", "Tm1"])
        & Q.field("consensus_nt").eq("acetylcholine")
    ),
    inputs=Q.field("type").eq("L2"),
    outputs=Q.field("type").eq("Tm1"),
    selection="strongest_input_to_output",
    max_inputs=32,
    max_outputs=64,
)

data_directory = next(
    root / "data" / "male-cns-v1.0"
    for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "data" / "male-cns-v1.0").is_dir()
)

core = FlyModel.from_directory(
    data_directory,
    circuit=circuit,
    sign_policy={"acetylcholine": 1},
    config=FlyConfig(propagation_steps=8),
)

model = FlyDataClassifier(
    core,
    num_features=x_train.shape[1],
    num_classes=2,
    rngs=nnx.Rngs(42),
)

In [ ]:
@nnx.jit
def train_step(model, optimizer, features, labels):
    def loss_fn(candidate):
        logits = candidate(features).logits
        return optax.softmax_cross_entropy_with_integer_labels(logits, labels).mean()

    loss, gradients = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, gradients)
    return loss


@nnx.jit
def evaluate_accuracy(model, features, labels):
    logits = model(features).logits
    return jnp.mean(jnp.argmax(logits, axis=-1) == labels)

In [ ]:
optimizer = nnx.Optimizer(model, optax.adam(1e-2), wrt=nnx.Param)

train_features = jnp.asarray(x_train)
train_labels = jnp.asarray(y_train)
test_features = jnp.asarray(x_test)
test_labels = jnp.asarray(y_test)
losses = []
train_accuracies = []
test_accuracies = []


trackio.init(
    project="flyx-make-moons",
    config={
        "dataset": "make_moons",
        "epochs": 500,
        "learning_rate": 1e-2,
        "num_neurons": core.num_neurons,
        "propagation_steps": core.config.propagation_steps,
        "seed": 42,
    },
)

In [ ]:
progress = trange(500, desc="Training")
for epoch in progress:
    loss = float(train_step(model, optimizer, train_features, train_labels))
    train_accuracy = float(evaluate_accuracy(model, train_features, train_labels))
    test_accuracy = float(evaluate_accuracy(model, test_features, test_labels))
    losses.append(loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)
    trackio.log(
        {
            "epoch": epoch + 1,
            "train/loss": loss,
            "train/accuracy": train_accuracy,
            "test/accuracy": test_accuracy,
        },
        step=epoch + 1,
    )
    progress.set_postfix(
        loss=f"{loss:.4f}",
        train_accuracy=f"{train_accuracy:.3f}",
        test_accuracy=f"{test_accuracy:.3f}",
    )

trackio.finish()